# Amazon-Reviews-2023 → ABSA smartphone corpus

Builds a **real** research corpus from [`McAuley-Lab/Amazon-Reviews-2023`](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023) (Cell Phones & Accessories).

Follows your methodology **Step 1** (preprocess) here in Colab, then you import the files into the local app for Steps 2–6 (segment → aspect extraction → sentiment → aggregate → recommend).

**No demo / sample phones.** Exact review text from the Hugging Face dataset is kept in `reviews.jsonl` (`body` field).

### Outputs (download these)
- `phones.jsonl`
- `reviews.jsonl`
- `manifest.json`

## 0) Install + clone the project
Uses your GitHub repo so the same filters/preprocess code run here and locally.

In [ ]:
!pip -q install datasets pyarrow pandas tqdm

# Clone your repo (public)
!rm -rf SmartPhone-Recommendation-System
!git clone --depth 1 https://github.com/KavishaLP/SmartPhone-Recommendation-System.git
%cd SmartPhone-Recommendation-System
!pip -q install -r requirements.txt

## 1) Configure corpus size
Start small to verify, then increase. Bigger `MAX_REVIEW_SCAN` finds more reviews for your ASINs (the review JSONL is huge).

In [ ]:
MAX_PHONES = 40          # how many real smartphones to keep
MAX_REVIEWS = 100        # exact reviews per phone (cap)
MIN_RATINGS = 500        # only products with at least this many Amazon ratings
MAX_REVIEW_SCAN = 2_000_000  # how far to scan the review stream
BRANDS = ["samsung", "apple", "google", "xiaomi", "oneplus", "motorola"]  # or [] for any brand
OUT = "hf_corpus"

print(dict(MAX_PHONES=MAX_PHONES, MAX_REVIEWS=MAX_REVIEWS, MIN_RATINGS=MIN_RATINGS,
           MAX_REVIEW_SCAN=MAX_REVIEW_SCAN, BRANDS=BRANDS, OUT=OUT))

## 2) Build corpus (meta → exact reviews → Step 1 preprocess)

This streams:
- meta parquet: `raw_meta_Cell_Phones_and_Accessories`
- reviews jsonl: `raw/review_categories/Cell_Phones_and_Accessories.jsonl`

Accessories are filtered out. Review `body` is the **exact** dataset text; `cleaned_body` is methodology Step 1.

In [ ]:
import sys
from pathlib import Path

brand_args = []
for b in BRANDS:
    brand_args += ["--brand", b]

cmd = [
    sys.executable, "scripts/build_hf_corpus.py",
    "--max-phones", str(MAX_PHONES),
    "--max-reviews", str(MAX_REVIEWS),
    "--min-ratings", str(MIN_RATINGS),
    "--max-review-scan", str(MAX_REVIEW_SCAN),
    "--out", OUT,
] + brand_args

print("Running:", " ".join(cmd))
!{sys.executable} scripts/build_hf_corpus.py --max-phones {MAX_PHONES} --max-reviews {MAX_REVIEWS} --min-ratings {MIN_RATINGS} --max-review-scan {MAX_REVIEW_SCAN} --out {OUT} {' '.join(brand_args)}

## 3) Inspect + download

In [ ]:
import json
from pathlib import Path
import pandas as pd

out = Path(OUT)
manifest = json.loads((out / "manifest.json").read_text())
print(json.dumps(manifest["counts"], indent=2))
print("\nPhones:")
for p in manifest["phones"]:
    print(f"  {p['asin']:12}  {p['usable_reviews']:4} usable  {p['name']}")

reviews = [json.loads(l) for l in (out / "reviews.jsonl").read_text().splitlines() if l]
df = pd.DataFrame(reviews)
print("\nSample exact review bodies:")
display(df.loc[df["usable"], ["parent_asin", "rating", "body"]].head(5))

In [ ]:
# Zip for download to your PC
!zip -r hf_corpus.zip {OUT}
from google.colab import files
files.download("hf_corpus.zip")

## 4) On your PC (after download)

Unzip into the project, wipe demo/sample data, import the real corpus, run Steps 2–6:

```powershell
cd "d:\4TH YEAR\RESEARCH SETUP\smartphone recommendation"
# unzip hf_corpus.zip so you have .\hf_corpus\phones.jsonl etc.

python run.py reset-corpus --yes
python run.py import-corpus .\hf_corpus --replace --analyze
python run.py serve
```

Then open http://127.0.0.1:8000/ui/ and use **Recommend**.

### Methodology mapping
| Step | Where it runs |
|---|---|
| 1 Preprocess (dedupe, spam/empty, HTML, normalise, language) | This Colab / `build_hf_corpus.py` |
| 2 Sentence segmentation | `import-corpus --analyze` |
| 3 Aspect extraction | `import-corpus --analyze` (lexicon now; set `LLM_API_KEY` for LLM) |
| 4 Sentiment classification | same |
| 5 Structured aspect-sentiment rows | SQLite `aspect_sentiments` |
| 6 Aspect score aggregation | SQLite `aspect_scores` → Recommend UI |